In [ ]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers trl pyyaml -q
print("Deps installed")


In [ ]:
# Cell 2: Clone CogMem + load tasks
REPO_BRANCH = "master"
!if [ ! -d /notebooks/CogMem/.git ]; then git clone --branch {REPO_BRANCH} --single-branch https://github.com/tungooxx/CogMem.git /notebooks/CogMem; else cd /notebooks/CogMem && git fetch origin && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}; fi
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
import json
from pathlib import Path
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
TASK_LIMIT = 834

if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "canonical_solution": item.get("canonical_solution", ""),
            "entry_point": item.get("entry_point", ""),
            "libs": item.get("libs", []),
        })
    with open(TASKS_PATH, "w") as f:
        for task in tasks:
            f.write(json.dumps(task) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

if TASK_LIMIT:
    tasks = tasks[:TASK_LIMIT]

print(Path('/notebooks/CogMem').resolve())
!cd /notebooks/CogMem && git branch --show-current && git rev-parse --short HEAD
print("Tasks loaded:", len(tasks))


In [ ]:
# Cell 3: Prepare split manifest + new-architecture configs
from cogmem.config import CogMemConfig
from cogmem.consolidation.experiment import (
    NewArchitectureExperimentConfig,
    prepare_new_arch_task_split,
    load_new_arch_runtime,
    _release_new_arch_runtime,
    run_new_arch_episode_collection,
    build_new_arch_skill_cards,
    compare_new_arch_routes,
    debug_episode_router_comparison,
    reset_route_runtime_utility,
    inspect_promoted_skill_routing,
    run_new_arch_qstar_cycle,
)

RESET_COLLECTION_PROGRESS = False
RESET_ROUTE_UTILITY = False  # Cell 9 benchmarking is read-only; do not clear utility unless starting a new study
RUN_QSTAR_TRAINING = False

NOTEBOOK_CONFIG = NewArchitectureExperimentConfig(
    experiment_dir="/notebooks/cogmem_new_architecture",
    manifest_path="/notebooks/cogmem_new_architecture/bigcodebench_manifest.json",
    memory_bank_path="/notebooks/cogmem_new_architecture/memory_bank.json",
    skills_path="/notebooks/cogmem_new_architecture/skill_cards.json",
    model_name="Qwen/Qwen2.5-3B-Instruct",
    task_limit=TASK_LIMIT,
    max_attempts=3,
    temperature=0.0,
)

split_result = prepare_new_arch_task_split(tasks, config=NOTEBOOK_CONFIG)
manifest = split_result["manifest"]
train_tasks = split_result["train_tasks"]
dev_tasks = split_result["dev_tasks"]
test_tasks = split_result["test_tasks"]

COGMEM_CONFIG = CogMemConfig(
    project_dir="/notebooks/CogMem",
    bigcodebench_memory_bank=NOTEBOOK_CONFIG.memory_bank_path,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    adapters_dir="/notebooks/cogmem_new_architecture/adapters",
    adapter_registry_path="/notebooks/cogmem_new_architecture/adapters/registry.json",
    experiments_dir="/notebooks/cogmem_new_architecture/experiments",
    logs_dir="/notebooks/cogmem_new_architecture/logs",
    skills_dir="/notebooks/cogmem_new_architecture/skills",
    active_model_hf=NOTEBOOK_CONFIG.model_name,
    base_model=NOTEBOOK_CONFIG.model_name,
    quantization_bits=0,  # Disable bitsandbytes for stable benchmark runs.
    use_dora=False,
    generator_rank=8,
    generator_alpha=16,
    verifier_rank=8,
    verifier_alpha=16,
    generator_batch_size=1,
    generator_max_seq_length=2048,
    generator_sft_epochs=3,
    generator_dpo_epochs=2,
    verifier_epochs=2,
    min_dpo_pairs=24,
    skill_min_distinct_tasks=2,
    skill_curriculum_examples_per_card=0,
    skill_retrieval_min_score=4.5,
    skill_retrieval_strict_min_score=6.0,
    skill_retrieval_min_promoted_families_for_broad_match=3,
    skill_runtime_disable_min_retrieved=3,
    skill_runtime_disable_min_hurt=1,
    skill_runtime_disable_hurt_rate=0.10,
    episode_summary_max_code_lines=8,
    episode_retrieval_allow_first_attempt=False,
    episode_first_attempt_min_score=10.0,
    episode_retrieval_min_score=7.0,
    episode_retrieval_retry_min_score=5.5,
    episode_runtime_disable_min_retrieved=1,
    episode_runtime_disable_min_hurt=1,
    episode_runtime_disable_hurt_rate=0.10,
    episode_runtime_neutral_downweight_min_retrieved=3,
    episode_runtime_neutral_downweight=3.0,
    blocked_episode_ids=[
        "bigcode_BigCodeBench_1_1776510708",
        "bigcode_BigCodeBench_339_1776515442",
    ],
    episode_route_eval_max_retrievals_per_episode=3,
    episode_route_eval_diversity_penalty=2.0,
    skill_induction_require_positive_episode_utility=True,
    skill_route_promotion_min_tasks=4,
    skill_route_promotion_min_delta_passed=2,
    skill_route_promotion_max_regression_rate=0.05,
    min_promoted_skills_for_adapter=3,
    min_skill_families_for_adapter=2,
    min_skill_training_pairs_for_adapter=48,
    skill_route_gate_task_limit=30,
    skill_route_gate_min_delta_passed=1,
    skill_route_gate_max_regression_rate=0.10,
    skill_route_gate_require_router_beats_episode=True,
    skill_route_gate_episode_delta_min_passed=0,
    skill_route_gate_episode_max_regression_rate=0.10,
    allowed_manifest_ids=[manifest["manifest_id"]],
    require_manifest_match=True,
    bigcodebench_eval_label="bigcodebench_cl",
)

print("Manifest:", manifest["manifest_id"])
print("Train tasks:", len(train_tasks))
print("Dev tasks  :", len(dev_tasks))
print("Test tasks :", len(test_tasks))
print("Memory bank:", NOTEBOOK_CONFIG.memory_bank_path)
print("Skill cards:", NOTEBOOK_CONFIG.skills_path)


In [ ]:
# Cell 4: Load local HF runtime for episode collection
import torch

base_model, tokenizer, llm_client = load_new_arch_runtime(
    model_name=NOTEBOOK_CONFIG.model_name,
    quantization_bits=COGMEM_CONFIG.quantization_bits,
)

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Runtime loaded. Free VRAM: {free:.1f} GB")


In [ ]:
# Cell 5: Collect typed episodes on the train split
collection_result = run_new_arch_episode_collection(
    train_tasks,
    llm_client,
    config=NOTEBOOK_CONFIG,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    reset_progress=RESET_COLLECTION_PROGRESS,
    verbose=True,
)

print()
print("=" * 60)
print("NEW-ARCH EPISODE COLLECTION COMPLETE")
print("Tasks processed    :", collection_result["tasks_processed"])
print("New episodes       :", collection_result["new_episodes"])
print("Episodes total     :", collection_result["episodes_total"])
print("New successes      :", collection_result["successes_this_run"])
print("Elapsed (min)      :", round(collection_result["elapsed_minutes"], 1))
print("Progress file      :", collection_result["progress_path"])
print("Memory bank path   :", collection_result["memory_bank_path"])
print("Summary metrics    :", collection_result["summary_metrics"])


In [ ]:
# Cell 6: Inspect typed episodic memory bank
from collections import Counter
from cogmem.memory.memory_bank import MemoryBank

bank = MemoryBank.load(NOTEBOOK_CONFIG.memory_bank_path)
metrics = bank.summary_metrics()
print("Episodes:", len(bank))
print("Summary metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

task_type_counts = Counter(ep.get("task_type", "general") for ep in bank)
error_family_counts = Counter(ep.get("error_family") or "None" for ep in bank)
split_counts = Counter(ep.get("split_name") or "unspecified" for ep in bank)

print()
print("Task types:")
for key, value in sorted(task_type_counts.items()):
    print(f"  {key}: {value}")

print()
print("Error families:")
for key, value in error_family_counts.most_common(10):
    print(f"  {key}: {value}")

print()
print("Split counts:")
for key, value in sorted(split_counts.items()):
    print(f"  {key}: {value}")


In [ ]:
# Cell 7: Build + validate procedural skill cards
skill_result = build_new_arch_skill_cards(
    NOTEBOOK_CONFIG.memory_bank_path,
    COGMEM_CONFIG,
    skills_path=NOTEBOOK_CONFIG.skills_path,
)

print("Episodes total     :", skill_result["episodes_total"])
print("Eligible episodes  :", skill_result["eligible_episodes"])
print("Available episodes :", skill_result["available_episodes"])
print("Skill induction episodes:", skill_result["skill_induction_episodes"])
print("Positive-only induction:", skill_result["skill_induction_positive_only"])
if skill_result["skill_induction_positive_only"] and skill_result["skill_induction_episodes"] == 0:
    print("No positive route-utility episodes found yet; run Cell 9 to seed episodic wins before expecting skill cards.")
print("Holdout episodes   :", skill_result["holdout_episodes"])
print("Task type counts   :", skill_result["task_type_counts"])
print("Skill summary      :", skill_result["skill_summary"])
if RESET_ROUTE_UTILITY:
    reset_result = reset_route_runtime_utility(
        skill_result["skills_path"],
        NOTEBOOK_CONFIG.memory_bank_path,
    )
    print("Route utility reset:", reset_result)

import json
routing_debug = inspect_promoted_skill_routing(
    skill_result["skills_path"],
    dev_tasks,
    config=COGMEM_CONFIG,
    task_limit=5,
    skill_top_k=3,
)
print("Promoted skill routing debug:")
print(json.dumps(routing_debug, indent=2, ensure_ascii=False))
skill_status = skill_result["skill_summary"]
promoted_count = int(skill_status.get("promoted", 0) or 0)
scored_dev_tasks = [row for row in routing_debug.get("task_scores", []) if row.get("scores")]
skill_layer_empty = promoted_count == 0 or not scored_dev_tasks
print("Skill layer empty:", skill_layer_empty)
if promoted_count == 0:
    print("Skill debug: no promoted skills passed held-out route promotion.")
elif not scored_dev_tasks:
    print("Skill debug: promoted skills exist, but retrieval thresholds or task matching selected none on sampled dev tasks.")
else:
    print("Skill debug: promoted skills can be scored on sampled dev tasks; inspect Cell 9 utility for actual runtime value.")
print("Training pairs     :", skill_result["training_pairs"])
print("Preference pairs   :", skill_result["preference_pairs"])

from pathlib import Path
import shutil

snapshot_dir = Path(NOTEBOOK_CONFIG.experiment_dir) / "benchmark_snapshots" / "cell7_frozen"
snapshot_dir.mkdir(parents=True, exist_ok=True)
benchmark_memory_path = str(snapshot_dir / "memory_bank.json")
benchmark_skills_path = str(snapshot_dir / "skill_cards.json")
shutil.copy2(NOTEBOOK_CONFIG.memory_bank_path, benchmark_memory_path)
shutil.copy2(skill_result["skills_path"], benchmark_skills_path)
print("Frozen benchmark memory:", benchmark_memory_path)
print("Frozen benchmark skills:", benchmark_skills_path)

print()
print("Top skill cards:")
for row in skill_result["skill_rows"]:
    print("Skill:", row["skill_id"])
    print(
        "  status:", row["status"],
        "| task_type:", row["task_type"],
        "| domain:", row["domain"],
        "| error_family:", row["error_family"],
    )
    print(
        "  evidence:", row["source_episode_count"],
        "| distinct_tasks:", row["distinct_task_count"],
        "| matched:", row["matched_episodes"],
        "| matched_tasks:", row["matched_tasks"],
        "| confidence:", round(row["confidence"], 3),
        "| transfer:", round(row["transfer_gain"], 3),
        "| negative_transfer:", round(row["negative_transfer_rate"], 3),
        "| success_rate:", round(row["success_rate"], 3),
        "| route_test:", row.get("route_test_status"),
    )
    print(
        "  route_delta:", row.get("route_delta_passed"),
        "| route_regression:", row.get("route_regression_rate"),
    )
    print("  triggers:", row["triggers"])
    print("  activation_conditions:", row["activation_conditions"])
    print("  plan_steps:", row["plan_steps"])
    print("  stop_conditions:", row["stop_conditions"])
    print("  anti_patterns:", row["anti_patterns"])


In [ ]:
# Cell 8: Optional Q-STaR consolidation cycle (adapter training)
if RUN_QSTAR_TRAINING:
    import gc
    import torch

    base_model = None
    tokenizer = None
    llm_client = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    qstar_result = run_new_arch_qstar_cycle(
        NOTEBOOK_CONFIG.memory_bank_path,
        COGMEM_CONFIG,
        cycle=0,
        skill_cards_path=skill_result["skills_path"],
        eval_tasks=dev_tasks,
        eval_task_limit=COGMEM_CONFIG.skill_route_gate_task_limit,
    )
    print(json.dumps(qstar_result, indent=2, ensure_ascii=False))
else:
    print("RUN_QSTAR_TRAINING is False. Set it to True to train generator/verifier adapters.")
    print("Promoted skills available:", len(skill_result["promoted_skill_ids"]))
    print("Skill cards path:", skill_result["skills_path"])


In [ ]:
# Cell 9: Evaluate base, retrieved-skill, retrieved-episode, router, and adapter routes
EVAL_SPLIT = "dev"  # or "test"
EVAL_TASK_LIMIT = 30  # keep benchmark cheap; set to None for the full split
EVAL_BENCHMARK_REPEATS = 3
EVAL_MAX_ATTEMPTS = 2
EVAL_SKILL_TOP_K = COGMEM_CONFIG.skill_retrieval_top_k
EVAL_ADAPTER_PATH = None  # keep adapters off until base+router beats base

eval_tasks = dev_tasks if EVAL_SPLIT == "dev" else test_tasks
source_skill_cards_path = skill_result.get("skills_path") if "skill_result" in globals() else NOTEBOOK_CONFIG.skills_path
skill_cards_path = globals().get("benchmark_skills_path", source_skill_cards_path)
episode_memory_path = globals().get("benchmark_memory_path", NOTEBOOK_CONFIG.memory_bank_path)
adapter_path = EVAL_ADAPTER_PATH

import gc
import json
import shutil
import subprocess
import torch
from datetime import datetime
from pathlib import Path

if COGMEM_CONFIG.quantization_bits != 0:
    raise RuntimeError(
        f"Cell 9 expects plain-weight benchmarking with quantization_bits=0, got {COGMEM_CONFIG.quantization_bits}. "
        "Rerun Cell 3 from the latest notebook state before benchmarking."
    )

if any(globals().get(name) is not None for name in ("base_model", "tokenizer", "llm_client")):
    print("Releasing existing local runtime before benchmark...")
    _release_new_arch_runtime(
        globals().get("base_model"),
        globals().get("tokenizer"),
        globals().get("llm_client"),
    )
    base_model = None
    tokenizer = None
    llm_client = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def benchmark_free_vram_gb():
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.STDOUT,
        )
    except Exception as exc:
        print(f"nvidia-smi query failed: {type(exc).__name__}: {exc}")
        return None
    values = []
    for line in output.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            values.append(float(line) / 1024)
        except ValueError:
            pass
    return max(values) if values else None

free_vram_gb = benchmark_free_vram_gb()
if free_vram_gb is not None:
    print(f"Free VRAM before Cell 9 benchmark: {free_vram_gb:.2f} GB")
    if free_vram_gb < 8.0:
        raise RuntimeError(
            f"Only {free_vram_gb:.2f} GB VRAM is free before Cell 9. "
            "Restart the kernel or rerun Cell 4 only when you actually need the interactive runtime."
        )

benchmark_snapshot_id = globals().get("benchmark_snapshot_id") or datetime.utcnow().strftime("%Y%m%d_%H%M%S")
snapshot_dir = Path(NOTEBOOK_CONFIG.experiment_dir) / "benchmark_snapshots"
snapshot_dir.mkdir(parents=True, exist_ok=True)

if "benchmark_memory_path" not in globals():
    snapshot_memory_path = snapshot_dir / f"memory_bank_{benchmark_snapshot_id}.json"
    shutil.copy2(NOTEBOOK_CONFIG.memory_bank_path, snapshot_memory_path)
    benchmark_memory_path = str(snapshot_memory_path)
    episode_memory_path = benchmark_memory_path

if "benchmark_skills_path" not in globals() and source_skill_cards_path and Path(source_skill_cards_path).exists():
    snapshot_skills_path = snapshot_dir / f"skill_cards_{benchmark_snapshot_id}.json"
    shutil.copy2(source_skill_cards_path, snapshot_skills_path)
    benchmark_skills_path = str(snapshot_skills_path)
    skill_cards_path = benchmark_skills_path

print("Benchmark snapshot:", benchmark_snapshot_id)
print("Benchmark memory:", episode_memory_path)
print("Benchmark skills:", skill_cards_path)

eval_results = []
for benchmark_run in range(1, EVAL_BENCHMARK_REPEATS + 1):
    print(f"\n=== READ-ONLY BENCHMARK RUN {benchmark_run}/{EVAL_BENCHMARK_REPEATS} ===")
    eval_result = compare_new_arch_routes(
        eval_tasks,
        model_name=NOTEBOOK_CONFIG.model_name,
        skill_cards_path=skill_cards_path,
        episode_memory_path=episode_memory_path,
        adapter_path=adapter_path,
        skill_top_k=EVAL_SKILL_TOP_K,
        config=COGMEM_CONFIG,
        task_limit=EVAL_TASK_LIMIT,
        max_tokens=NOTEBOOK_CONFIG.max_tokens,
        temperature=0.0,
        max_attempts=EVAL_MAX_ATTEMPTS,
        eval_timeout=NOTEBOOK_CONFIG.eval_timeout,
        verbose=(benchmark_run == 1),
    )
    eval_results.append(eval_result)

eval_result = eval_results[-1]

def route_evidence(result, route_name, comparison_name):
    route_eval = result.get(route_name, {}) or {}
    comparison = result.get("comparisons", {}).get(comparison_name, {}) or {}
    selected_skills = route_eval.get("selected_skill_ids", {}) or {}
    selected_episodes = route_eval.get("selected_episode_ids", {}) or {}
    skill_utility = comparison.get("skill_utility", {}) or {}
    episode_utility = comparison.get("episode_utility", {}) or {}
    return {
        "selected_skills": selected_skills,
        "selected_episodes": selected_episodes,
        "skill_utility": skill_utility,
        "episode_utility": episode_utility,
        "has_skill_evidence": bool(selected_skills) and bool(skill_utility),
        "has_episode_evidence": bool(selected_episodes) and bool(episode_utility),
    }

def compact_run_summary(result):
    comparisons = result.get("comparisons", {})
    skill_evidence = route_evidence(result, "base_plus_skill", "base_plus_skill_vs_base")
    episode_evidence = route_evidence(result, "base_plus_episode", "base_plus_episode_vs_base")
    router_evidence = route_evidence(result, "base_plus_router", "base_plus_router_vs_base")
    router_vs_episode = comparisons.get("base_plus_router_vs_base_plus_episode", {}) or {}
    return {
        "base_passed": result["base"]["passed"],
        "base_plus_skill_passed": result.get("base_plus_skill", {}).get("passed"),
        "base_plus_episode_passed": result.get("base_plus_episode", {}).get("passed"),
        "base_plus_router_passed": result.get("base_plus_router", {}).get("passed"),
        "base_plus_skill_delta_passed": comparisons.get("base_plus_skill_vs_base", {}).get("delta_passed"),
        "base_plus_episode_delta_passed": comparisons.get("base_plus_episode_vs_base", {}).get("delta_passed"),
        "base_plus_router_delta_passed": comparisons.get("base_plus_router_vs_base", {}).get("delta_passed"),
        "router_vs_episode_delta_passed": router_vs_episode.get("delta_passed"),
        "router_vs_episode_regression_rate": router_vs_episode.get("regression_rate"),
        "skill_has_evidence": skill_evidence["has_skill_evidence"],
        "episode_has_evidence": episode_evidence["has_episode_evidence"],
        "router_has_evidence": router_evidence["has_skill_evidence"] or router_evidence["has_episode_evidence"],
    }

benchmark_run_summaries = [compact_run_summary(result) for result in eval_results]

def repeated_value(key):
    values = [row.get(key) for row in benchmark_run_summaries]
    return values[0] if values and all(value == values[0] for value in values) else None

trusted_route_signals = {
    "base_plus_skill": repeated_value("base_plus_skill_delta_passed") is not None and all(row["skill_has_evidence"] for row in benchmark_run_summaries),
    "base_plus_episode": repeated_value("base_plus_episode_delta_passed") is not None and all(row["episode_has_evidence"] for row in benchmark_run_summaries),
    "base_plus_router": repeated_value("base_plus_router_delta_passed") is not None and all(row["router_has_evidence"] for row in benchmark_run_summaries),
}
router_vs_episode = eval_result.get("comparisons", {}).get("base_plus_router_vs_base_plus_episode", {}) or {}
router_ready_vs_episode = (
    repeated_value("router_vs_episode_delta_passed") is not None
    and router_vs_episode.get("delta_passed", -10**9) >= COGMEM_CONFIG.skill_route_gate_episode_delta_min_passed
    and router_vs_episode.get("regression_rate", 1.0) <= COGMEM_CONFIG.skill_route_gate_episode_max_regression_rate
    and trusted_route_signals["base_plus_router"]
)

summary = {
    "split": EVAL_SPLIT,
    "task_count": eval_result["task_count"],
    "read_only_benchmark": True,
    "benchmark_quantization_bits": COGMEM_CONFIG.quantization_bits,
    "benchmark_repeats": EVAL_BENCHMARK_REPEATS,
    "benchmark_snapshot_id": benchmark_snapshot_id,
    "benchmark_memory_path": episode_memory_path,
    "benchmark_skills_path": skill_cards_path,
    "temporary_serving_baseline": "base_plus_episode",
    "trusted_route_signals": trusted_route_signals,
    "benchmark_run_summaries": benchmark_run_summaries,
    "router_ready_vs_episode": router_ready_vs_episode,
    "base_passed": eval_result["base"]["passed"],
    "base_pass_rate": eval_result["base"]["pass_rate"],
    "base_plus_skill_passed": eval_result.get("base_plus_skill", {}).get("passed"),
    "base_plus_skill_pass_rate": eval_result.get("base_plus_skill", {}).get("pass_rate"),
    "base_plus_skill_delta_passed": eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("delta_passed"),
    "base_plus_skill_regression_rate": eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("regression_rate"),
    "base_plus_episode_passed": eval_result.get("base_plus_episode", {}).get("passed"),
    "base_plus_episode_pass_rate": eval_result.get("base_plus_episode", {}).get("pass_rate"),
    "base_plus_episode_delta_passed": eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("delta_passed"),
    "base_plus_episode_regression_rate": eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("regression_rate"),
    "base_plus_router_passed": eval_result.get("base_plus_router", {}).get("passed"),
    "base_plus_router_pass_rate": eval_result.get("base_plus_router", {}).get("pass_rate"),
    "base_plus_router_delta_passed": eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("delta_passed"),
    "base_plus_router_regression_rate": eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("regression_rate"),
    "router_vs_episode_delta_passed": router_vs_episode.get("delta_passed"),
    "router_vs_episode_delta_pass_rate": router_vs_episode.get("delta_pass_rate"),
    "router_vs_episode_regression_rate": router_vs_episode.get("regression_rate"),
    "router_vs_episode_improved_task_ids": router_vs_episode.get("improved_task_ids", [])[:10],
    "router_vs_episode_regressed_task_ids": router_vs_episode.get("regressed_task_ids", [])[:10],
    "adapter_passed": eval_result.get("adapter", {}).get("passed"),
    "adapter_pass_rate": eval_result.get("adapter", {}).get("pass_rate"),
    "adapter_plus_router_passed": eval_result.get("adapter_plus_router", {}).get("passed"),
    "adapter_plus_router_pass_rate": eval_result.get("adapter_plus_router", {}).get("pass_rate"),
    "delta_passed": eval_result.get("delta_passed"),
    "delta_pass_rate": eval_result.get("delta_pass_rate"),
    "improved_task_ids": eval_result.get("improved_task_ids", [])[:10],
    "regressed_task_ids": eval_result.get("regressed_task_ids", [])[:10],
    "base_plus_skill_selected": eval_result.get("base_plus_skill", {}).get("selected_skill_ids", {}),
    "base_plus_episode_selected": eval_result.get("base_plus_episode", {}).get("selected_episode_ids", {}),
    "base_plus_router_routes": eval_result.get("base_plus_router", {}).get("selected_route_counts", {}),
    "adapter_plus_router_routes": eval_result.get("adapter_plus_router", {}).get("selected_route_counts", {}),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Base+skill utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_skill_vs_base", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
print("Base+episode utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_episode_vs_base", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))
print("Base+router skill utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
print("Base+router episode utility:")
print(json.dumps(eval_result.get("comparisons", {}).get("base_plus_router_vs_base", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))
print("Router vs episode baseline:")
print(json.dumps(router_vs_episode, indent=2, ensure_ascii=False))
if adapter_path:
    print("Adapter+router skill utility:")
    print(json.dumps(eval_result.get("comparisons", {}).get("adapter_plus_router_vs_adapter", {}).get("skill_utility", {}), indent=2, ensure_ascii=False))
    print("Adapter+router episode utility:")
    print(json.dumps(eval_result.get("comparisons", {}).get("adapter_plus_router_vs_adapter", {}).get("episode_utility", {}), indent=2, ensure_ascii=False))


In [ ]:
# Cell 10: Targeted router-loss debug for reset-sensitive runs
from pathlib import Path
import json
import shutil

DEBUG_TASK_IDS = ["BigCodeBench/26", "BigCodeBench/74", "BigCodeBench/86"]
debug_skill_cards_path = globals().get("benchmark_skills_path", skill_result.get("skills_path") if "skill_result" in globals() else NOTEBOOK_CONFIG.skills_path)
debug_episode_memory_path = globals().get("benchmark_memory_path", NOTEBOOK_CONFIG.memory_bank_path)
debug_output_dir = Path(NOTEBOOK_CONFIG.experiment_dir) / "debug_router_losses"

debug_report = debug_episode_router_comparison(
    DEBUG_TASK_IDS,
    tasks,
    model_name=NOTEBOOK_CONFIG.model_name,
    skill_cards_path=debug_skill_cards_path,
    episode_memory_path=debug_episode_memory_path,
    skill_top_k=EVAL_SKILL_TOP_K if "EVAL_SKILL_TOP_K" in globals() else COGMEM_CONFIG.skill_retrieval_top_k,
    config=COGMEM_CONFIG,
    max_tokens=NOTEBOOK_CONFIG.max_tokens,
    temperature=0.0,
    max_attempts=2,
    eval_timeout=NOTEBOOK_CONFIG.eval_timeout,
    output_dir=str(debug_output_dir),
    verbose=True,
)

# Also copy artifacts into the repo so they are easy to commit or download before Paperspace resets.
repo_docs_dir = Path("/notebooks/CogMem/docs") if Path("/notebooks/CogMem").exists() else Path("docs")
repo_docs_dir.mkdir(parents=True, exist_ok=True)
repo_json_path = repo_docs_dir / "router_loss_debug_latest.json"
repo_md_path = repo_docs_dir / "router_loss_debug_latest.md"
shutil.copy2(debug_report["json_path"], repo_json_path)
shutil.copy2(debug_report["markdown_path"], repo_md_path)

print(json.dumps(debug_report["summary"], indent=2, ensure_ascii=False))
for task in debug_report["tasks"]:
    print("\n", task["task_id"], "=>", task["diagnosis"])
    print("  episode:", task["base_plus_episode"].get("passed"), task["base_plus_episode"].get("retrieved_episode_id"), task["base_plus_episode"].get("error"))
    print("  router :", task["base_plus_router"].get("passed"), task["base_plus_router"].get("retrieved_episode_id"), task["base_plus_router"].get("retrieved_skill_ids"), task["base_plus_router"].get("error"))
    print("  router history:", json.dumps(task["base_plus_router"].get("retrieved_route_history", []), ensure_ascii=False))

print("\nSaved debug JSON:", debug_report["json_path"])
print("Saved debug MD  :", debug_report["markdown_path"])
print("Repo JSON copy  :", repo_json_path)
print("Repo MD copy    :", repo_md_path)
